In [2]:
import os
import re
import time
import requests
import xml.etree.ElementTree as ET
from tabulate import tabulate

# =========================
# CONFIG
# =========================

BASE = "data"
MD_DIR = os.path.join(BASE, "md")

os.makedirs(MD_DIR, exist_ok=True)

MAX_HTTP_RETRIES = 5


# =========================
# UTILS
# =========================

def normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def textify(elem):
    if elem is None:
        return ""

    parts = [elem.text or ""]
    for c in elem:
        parts.append(textify(c))
        if c.tail:
            parts.append(c.tail)

    return normalize("".join(parts))


def save_text(data, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(data)


# =========================
# PMC FETCH
# =========================

def doi_to_pmcid(doi):
    url = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"
    params = {"ids": doi, "format": "json"}

    for _ in range(MAX_HTTP_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=30)
            data = r.json()
            if "records" in data and data["records"]:
                return data["records"][0].get("pmcid")
        except:
            pass
        time.sleep(1)

    return None


def fetch_pmc_xml(pmcid):
    pmc_num = pmcid[3:]

    url = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"
    params = {
        "verb": "GetRecord",
        "identifier": f"oai:pubmedcentral.nih.gov:{pmc_num}",
        "metadataPrefix": "pmc",
    }

    for _ in range(MAX_HTTP_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 200:
                return r.text
        except:
            pass
        time.sleep(1)

    return None


# =========================
# TABLE PARSER (XML → MD)
# =========================

def parse_table(table):
    if table is None:
        return None

    rows = []

    thead = table.find(".//{*}thead")
    if thead is not None:
        for tr in thead.findall("{*}tr"):
            rows.append([textify(td) for td in tr.findall("{*}th")])

    tbody = table.find(".//{*}tbody")
    if tbody is not None:
        for tr in tbody.findall("{*}tr"):
            rows.append([textify(td) for td in tr.findall("{*}td")])

    if not rows:
        return None

    header = rows[0]

    md = []
    md.append("| " + " | ".join(header) + " |")
    md.append("| " + " | ".join("---" for _ in header) + " |")

    for r in rows[1:]:
        if len(r) < len(header):
            r += [""] * (len(header) - len(r))
        md.append("| " + " | ".join(r) + " |")

    return "\n".join(md)


# =========================
# XML → MARKDOWN
# =========================

def parse_article(xml):
    root = ET.fromstring(xml)
    article = root.find(".//{*}article")

    if article is None:
        return None

    out = []

    title = article.find(".//{*}article-title")
    if title is not None:
        out.append(f"# {textify(title)}\n")

    body = article.find("{*}body")
    if body is not None:
        for sec in body.findall(".//{*}sec"):

            sec_title = sec.find("{*}title")
            if sec_title is not None:
                out.append(f"\n## {textify(sec_title)}\n")

            # TEXT
            for p in sec.findall("{*}p"):
                txt = textify(p)
                if txt:
                    out.append(txt + "\n")

            # FIGURES
            for fig in sec.findall(".//{*}fig"):
                label = fig.find("{*}label")
                caption = fig.find(".//{*}caption/p")
                graphic = fig.find(".//{*}graphic")

                if graphic is not None:
                    href = graphic.attrib.get("{http://www.w3.org/1999/xlink}href")
                    if href:
                        out.append(f"\n![{textify(label)} {textify(caption)}]({href})\n")

            # TABLES
            for tbl_wrap in sec.findall(".//{*}table-wrap"):

                label = tbl_wrap.find("{*}label")
                caption = tbl_wrap.find(".//{*}caption")
                table = tbl_wrap.find("{*}table")

                if label is not None:
                    out.append(f"\n**{textify(label)}**\n")

                if caption is not None:
                    out.append(f"*{textify(caption)}*\n")

                md_table = parse_table(table)
                if md_table:
                    out.append(md_table + "\n")

    return "\n".join(out)


# =========================
# CLEAN MARKDOWN (🔥 NOWE)
# =========================

def clean_markdown(md_text):
    lines = md_text.split("\n")

    cleaned = []
    current_section = None
    current_content = []

    seen_sections = set()

    def flush():
        nonlocal current_section, current_content

        if current_section:
            content = "\n".join(current_content).strip()

            if content:
                key = normalize(current_section.lower() + content[:200])

                if key not in seen_sections:
                    seen_sections.add(key)
                    cleaned.append(current_section)
                    cleaned.append(content)
                    cleaned.append("")

        current_section = None
        current_content = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("## "):
            flush()
            current_section = stripped
        else:
            if current_section:
                current_content.append(line)
            else:
                cleaned.append(line)

    flush()

    return "\n".join(cleaned)


# =========================
# MD TABLE PARSER
# =========================

def parse_md_table(text):
    lines = [l.strip() for l in text.split("\n") if l.strip().startswith("|")]

    if len(lines) < 2:
        return None

    def split(line):
        cells = [x.strip() for x in line.split("|")]
        return [c for c in cells if c]

    rows = [split(l) for l in lines]

    header = rows[0]
    data = []

    for r in rows[1:]:
        if all(re.fullmatch(r"-+", c) for c in r):
            continue
        data.append(r)

    return header, data


# =========================
# PARSE MARKDOWN (UI)
# =========================

def parse_markdown(md_text):
    title = ""
    sections = []
    current = None
    current_table = []

    def flush_table():
        nonlocal current_table, current
        if current and current_table:
            idx = len(current["tables"])
            current["tables"].append("\n".join(current_table))
            current["text"] += f" [[TABLE_{idx}]] "
            current_table.clear()

    for line in md_text.split("\n"):
        stripped = line.strip()

        if not stripped:
            flush_table()
            continue

        if stripped.startswith("# ") and not title:
            title = stripped[2:]
            continue

        if stripped.startswith("## "):
            flush_table()
            if current:
                sections.append(current)

            heading = stripped[3:]

            if heading.lower() in ["authors", "emails"]:
                current = None
                continue

            current = {"heading": heading, "text": "", "tables": []}
            continue

        if stripped.startswith("|"):
            current_table.append(stripped)
            continue

        flush_table()

        if current:
            current["text"] += " " + line

    flush_table()

    if current:
        sections.append(current)

    return title, sections


# =========================
# VIEW
# =========================

def interactive_view(md_path, authors, emails):
    with open(md_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    title, sections = parse_markdown(md_text)

    print(f"\nDocument: {title}\n")

    print("Authors:")
    for a in authors:
        print(f"- {a}")

    print("\nEmails:")
    for i, e in enumerate(emails):
        print(f"- {e}" + (" (corresponding)" if i == 0 else ""))

    while True:
        print("\nAvailable sections:\n")
        for i, sec in enumerate(sections):
            print(f"{i+1}. {sec['heading']}")

        choice = input("\nSelect section: ")

        if choice in ["n", "q"]:
            break

        if not choice.isdigit():
            continue

        sec = sections[int(choice)-1]

        print(f"\n--- {sec['heading']} ---\n")
        print(sec["text"])

        # TABLES
        for t in sec["tables"]:
            parsed = parse_md_table(t)
            if parsed:
                h, r = parsed
                print("\n[Table]\n")
                print(tabulate(r, headers=h, tablefmt="grid"))

        # IMAGES
        imgs = re.findall(r'!\[.*?\]\((.*?)\)', sec["text"])
        for i, img in enumerate(imgs):
            print(f"\n[Image {i+1}] {img}")

        again = input("\nAnything else? (y/n): ")
        if again != "y":
            break


# =========================
# PROCESS
# =========================

def process_pubmed(doi):
    pmcid = doi_to_pmcid(doi)
    if not pmcid:
        return None

    xml = fetch_pmc_xml(pmcid)
    if not xml:
        return None

    root = ET.fromstring(xml)

    authors = []
    emails = []

    for c in root.findall(".//{*}contrib[@contrib-type='author']"):
        g = c.find(".//{*}given-names")
        s = c.find(".//{*}surname")
        if g is not None and s is not None:
            authors.append(normalize(f"{g.text} {s.text}"))

        e = c.find(".//{*}email")
        if e is not None:
            emails.append(normalize(e.text))

    md = parse_article(xml)

    cleaned_md = clean_markdown(md)

    path = os.path.join(MD_DIR, f"{pmcid}.md")
    save_text(cleaned_md, path)

    return {
        "doi": doi,
        "md": path,
        "authors": list(dict.fromkeys(authors)),
        "emails": list(dict.fromkeys(emails))
    }


# =========================
# MAIN
# =========================

if __name__ == "__main__":

    dois = input("DOIs: ").split(",")

    processed = []
    success = 0
    failed = 0

    for d in dois:
        r = process_pubmed(d.strip())
        if r:
            processed.append(r)
            success += 1
        else:
            failed += 1

    print("\n==== SUMMARY ====")
    print(f"Processed successfully: {success}")
    print(f"Failed: {failed}")
    print(f"Total: {len(dois)}")

    while True:
        for i, p in enumerate(processed):
            print(f"{i+1}. {p['doi']}")

        c = input("Select: ")
        if c in ["q", "n"]:
            break

        interactive_view(
            processed[int(c)-1]["md"],
            processed[int(c)-1]["authors"],
            processed[int(c)-1]["emails"]
        )


==== SUMMARY ====
Processed successfully: 1
Failed: 0
Total: 1
1. 10.1111/pai.70147

Document: Pre‐chewing for weaning: A traditional method for allergy prevention? Rationale, study design, and methods of the open‐label trial—Solids‐by‐Kiss

Authors:
- Birgit Ahrens
- Lara Meixner
- Meral Sturmfels
- Birgit Kalb
- Anna Fischl
- Falk Schwendicke
- Katharina Blumchen
- Andreas Fickenscher
- Laura Schäfer
- Thomas Holzhauser
- Sabine Schnadt
- Kirsten Beyer

Emails:
- ahrens@med.uni-frankfurt.de (corresponding)

Available sections:

1. INTRODUCTION
2. Background and rationale
3. Microbial diversity supports inflammatory resilience
4. Early dietary intervention in the development of food allergies
5. Objectives
6. Trial design
7. METHODS: PARTICIPANTS, INTERVENTIONS, AND OUTCOMES
8. Study setting
9. Eligibility criteria
10. Intervention
11. Intervention description
12. Strategies to improve adherence to interventions and criteria for discontinuing or modifying allocated interventions
13. 

ValueError: invalid literal for int() with base 10: 'm'